# Research one node: EMA spread

Hypothesis: a positive fast/slow EMA spread predicts positive subsequent returns. This is a workflow example, not an established edge or paper replication. See `examples/EMA_RESEARCH.md` for sources and limitations.

Run in QuantConnect Research with this repository's Python modules available. Create fresh node/provider instances for each experiment. QC owns indicators and rolling windows.

Cloud: use the copy uploaded by `sugarloaf upload --adapter quantconnect --manifest deployments/quantconnect/ema.toml`. The adapter admits split/dividend facts at effective boundaries and resets affected resources before post-action reads. Historical receipt times equal observation ends by assumption; native numerical parity and adjustment-vintage correctness remain unverified.

In [ ]:
from datetime import UTC, datetime, timedelta

from AlgorithmImports import QuantBook

from examples.ema import EMAConfig, EMAScore
from sugarloaf.adapters.quantconnect import QuantBookServices
from sugarloaf.core.primitives import InstrumentId
from sugarloaf.research import ReturnOutcomeResolver, evaluate, run_node

instrument = InstrumentId("USA:SPY")
config = EMAConfig(fast=5, slow=20, interval=timedelta(days=1), horizon=1)
start = datetime(2025, 1, 6, tzinfo=UTC)
end = datetime(2025, 1, 18, tzinfo=UTC)
qb = QuantBook()
services = QuantBookServices(qb, tickers={instrument: "SPY"})

In [ ]:
run = run_node(
    lambda: EMAScore(instrument, config),
    services,
    start=start,
    end=end,
    configuration={
        "fast": config.fast,
        "slow": config.slow,
        "interval_seconds": config.interval.total_seconds(),
        "horizon_sessions": config.horizon,
    },
)
print("Forecasts:", len(run.archive))
print("Readiness diagnostics:", list(run.runtime.diagnostics)[-5:])
run.provenance

Resolve outcomes only after generating forecasts. The resolver reads exact endpoint prices through a separate history request; future labels never enter the node's input provider. This example uses a horizon of one trading session, so a Friday forecast resolves at Monday's close. An endpoint with no price (for example an unscheduled market closure) remains explicitly unavailable.

The archive preserves revised/expired forecasts. Grouped association diagnostics are descriptive; overlapping horizons are not independent trials. Use separate chronological development/validation periods before interpreting predictive value.

In [ ]:
as_of = datetime(2025, 1, 22, tzinfo=UTC)
resolver = ReturnOutcomeResolver(
    services.anchor_observation, provenance={"provider": "QuantConnect historical prices"}
)
report = evaluate(run.archive, resolver, as_of=as_of)
print("Coverage:", report.coverage)
report.summary()

In [ ]:
# Inspect actual predictions and labels, including unresolved outcomes.
[(row.forecast.view.metadata.view_id, row.status, row.metrics) for row in report.rows[:5]]

Record the source, configuration, dates, results, failed variants and limitations beside the node. To test strategy contribution, use `examples.ema.build_graph` with the same `EMAConfig`; use the `deployments/quantconnect/ema.toml` deployment manifest.

The notebook ships without generated outputs; see `docs/VALIDATION.md` for which local and Cloud checks have actually run.